In [1]:
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import requests
import random
import json
import os
import logging
from logging.handlers import RotatingFileHandler

class PromptFormatter:
    """Clase para manejar diferentes formatos de prompt según el modelo"""
    
    def __init__(self, logger):
        self.logger = logger
    
    @staticmethod
    def format_mistral_prompt(messages):
        """Convierte mensajes al formato de prompt para Mistral"""
        formatted_prompt = ""
        system_content = next((m["content"] for m in messages if m["role"] == "system"), "")
        
        # Comenzar con el contenido del sistema y el primer mensaje del usuario
        user_messages = [m for m in messages if m["role"] == "user"]
        if user_messages:
            formatted_prompt = f"<s>[INST] {system_content}\n{user_messages[0]['content']} [/INST]"
            
            # Agregar el resto de los mensajes en el formato correcto
            for i in range(1, len(user_messages)):
                formatted_prompt += f" [INST] {user_messages[i]['content']} [/INST]"
                
        return formatted_prompt

    @staticmethod
    def parse_mistral_response(response):
        """Parsea la respuesta de Mistral"""
        if response and 'choices' in response:
            # Para el endpoint /v1/completions
            if 'text' in response['choices'][0]:
                return response['choices'][0]['text'].strip()
            # Para compatibilidad con otros endpoints
            elif 'message' in response['choices'][0]:
                return response['choices'][0]['message']['content'].strip()
        return None

class MistralPromptFormatter:
    """Formateador especializado para prompts de Mistral con enfoque en español"""
    
    def __init__(self, logger):
        self.logger = logger
        self.language_instruction = """
        A partir de este momento, asume el papel de un asistente con dominio perfecto del español. 
        Todas tus respuestas deben estar en español, utilizando vocabulario, expresiones y estructuras 
        gramaticales propias del idioma. Adapta tu estilo según el contexto, manteniendo la coherencia 
        lingüística y cultural en todas tus interacciones. Los ejemplos técnicos y código deben tener 
        comentarios en español.
        """.strip()

    def format_system_instruction(self, base_instruction, personality_config):
        """Formatea la instrucción del sistema combinando la base con la configuración de personalidad"""
        return f"<s>[INST] {self.language_instruction}\n{base_instruction} {personality_config} [/INST]"

    def format_user_message(self, message, is_first=False):
        """Formatea un mensaje del usuario"""
        reminder = " Recuerda mantener todas tus respuestas en español." if not is_first else ""
        return f"[INST]El usuario indica: {message}{reminder}[/INST]"

    def format_assistant_response(self, response, is_first=False):
        """Formatea una respuesta del asistente"""
        suffix = " </s>" if is_first else ""
        return f"{response}{suffix}"

    def format_conversation(self, messages, personality_config):
        """Formatea una conversación completa para Mistral"""
        formatted_prompt = ""
        
        # Primeros dos mensajes del sistema (si existen)
        if len(messages) > 2:
            formatted_prompt = self.format_system_instruction(
                messages[0]["content"], 
                personality_config
            )

        # Primer intercambio usuario-asistente
        if len(messages) >= 4:
            formatted_prompt += self.format_user_message(messages[2]["content"], is_first=True)
            formatted_prompt += self.format_assistant_response(messages[3]["content"], is_first=True)

        # Resto de la conversación
        if len(messages) >= 5:
            for i in range(4, len(messages)):
                if messages[i]["role"] == "user":
                    formatted_prompt += self.format_user_message(messages[i]["content"])
                else:
                    formatted_prompt += self.format_assistant_response(messages[i]["content"])

        return formatted_prompt

class ConversationGovernance:
    """
    Clase para manejar el gobierno y las reglas de la conversación.
    Define y valida las rutas conversacionales permitidas.
    """
    def __init__(self, types_config, stages_config):
        self.types_config = types_config
        self.stages_config = stages_config
        self.transition_rules = self._initialize_transition_rules()
        
    def _initialize_transition_rules(self):
        """
        Define las reglas de transición entre tipos y etapas de conversación.
        Retorna un diccionario con las transiciones permitidas.
        """
        return {
            # Reglas para clarification_and_framing
            'clarification_and_framing': {
                'initiation': ['development', 'practical_assistance'],
                'development': ['consolidation', 'informative_analytical'],
                'consolidation': ['transition', 'structured_emotional_support'],
                'transition': ['initiation']  # Permite reiniciar el ciclo
            },
            # Reglas para practical_assistance
            'practical_assistance': {
                'initiation': ['development', 'informative_analytical'],
                'development': ['consolidation', 'clarification_and_framing'],
                'consolidation': ['transition'],
                'transition': ['initiation', 'clarification_and_framing']
            },
            # Reglas para informative_analytical
            'informative_analytical': {
                'initiation': ['development', 'practical_assistance'],
                'development': ['consolidation', 'clarification_and_framing'],
                'consolidation': ['transition'],
                'transition': ['initiation']
            },
            # Reglas para structured_emotional_support
            'structured_emotional_support': {
                'initiation': ['development'],
                'development': ['consolidation', 'clarification_and_framing'],
                'consolidation': ['transition'],
                'transition': ['initiation', 'practical_assistance']
            }
        }

    def predict_next_state(self, current_type, current_stage, message_intent):
        """
        Predice el siguiente estado de la conversación basado en las reglas y el intent.
        
        Args:
            current_type (str): Tipo actual de la conversación
            current_stage (str): Etapa actual de la conversación
            message_intent (dict): Intención detectada en el mensaje
            
        Returns:
            tuple: (next_type, next_stage, confidence)
        """
        # Obtener posibles transiciones
        possible_transitions = self.transition_rules.get(current_type, {}).get(current_stage, [])
        
        if not possible_transitions:
            return current_type, 'initiation', 0.5
            
        # Analizar el intent para determinar la mejor transición
        next_state = self._analyze_intent_for_transition(
            possible_transitions,
            message_intent,
            current_type,
            current_stage
        )
        
        return next_state

    def _analyze_intent_for_transition(self, possible_transitions, intent, current_type, current_stage):
        """
        Analiza el intent para determinar la mejor transición posible.
        
        Args:
            possible_transitions (list): Lista de transiciones permitidas
            intent (dict): Intención detectada en el mensaje
            current_type (str): Tipo actual de la conversación
            current_stage (str): Etapa actual de la conversación
            
        Returns:
            tuple: (next_type, next_stage, confidence)
        """
        # Extraer información relevante del intent
        intent_type = intent.get('type', current_type)
        intent_confidence = intent.get('confidence', 0.5)
        
        # Si el intent sugiere un tipo diferente y está en las transiciones permitidas
        if intent_type in possible_transitions:
            next_type = intent_type
            next_stage = 'initiation'  # Nueva tipo comienza en initiation
            confidence = intent_confidence
        else:
            # Mantener el tipo actual pero avanzar en las etapas
            next_type = current_type
            next_stage = self._get_next_stage(current_stage)
            confidence = 0.7  # Confianza moderada en transiciones de etapa
            
        return next_type, next_stage, confidence

    def _get_next_stage(self, current_stage):
        """Determina la siguiente etapa en la secuencia natural"""
        stages = ['initiation', 'development', 'consolidation', 'transition']
        try:
            current_index = stages.index(current_stage)
            next_index = (current_index + 1) % len(stages)
            return stages[next_index]
        except ValueError:
            return 'initiation'

    def validate_transition(self, current_type, current_stage, proposed_type, proposed_stage):
        """
        Valida si una transición propuesta es permitida según las reglas.
        
        Args:
            current_type (str): Tipo actual de la conversación
            current_stage (str): Etapa actual de la conversación
            proposed_type (str): Tipo propuesto para la siguiente interacción
            proposed_stage (str): Etapa propuesta para la siguiente interacción
            
        Returns:
            tuple: (is_valid, confidence, alternative)
        """
        # Verificar si la transición está en las reglas
        allowed_transitions = self.transition_rules.get(current_type, {}).get(current_stage, [])
        
        if proposed_type in allowed_transitions:
            return True, 0.8, None
            
        # Si no está permitida, sugerir una alternativa válida
        alternative = None
        if allowed_transitions:
            alternative = (allowed_transitions[0], 'initiation')
            
        return False, 0.3, alternative

    def get_conversation_parameters(self, conv_type, stage):
        """
        Obtiene los parámetros específicos para un tipo y etapa de conversación.
        
        Args:
            conv_type (str): Tipo de conversación
            stage (str): Etapa de la conversación
            
        Returns:
            dict: Parámetros para la generación de la respuesta
        """
        type_config = self.types_config["types"].get(conv_type, {})
        stage_config = self.stages_config["stages"].get(stage, {})
        
        return {
            'system_template': type_config.get('system', ''),
            'focus_options': type_config.get('focus', []),
            'stage_focus': stage_config.get('focus', []),
            'max_tokens': {
                'initiation': 100,
                'development': 150,
                'consolidation': 200,
                'transition': 100
            }.get(stage, 150)
        }

class ConversationGenerator:
    def __init__(self, model, api_url, api_key, debug=False):
        self.model = model
        self.api_url = api_url
        self.api_key = api_key
        
        # Crear directorios necesarios
        self.base_dir = os.getcwd()
        self.logs_dir = os.path.join(self.base_dir, 'logs')
        self.generated_dir = os.path.join(self.base_dir, 'generated')
        self.config_dir = os.path.join(self.base_dir, 'config')
        
        # Crear directorios si no existen
        for directory in [self.logs_dir, self.generated_dir, self.config_dir]:
            os.makedirs(directory, exist_ok=True)
        
        # Configuración de logging
        self.setup_logging(debug)
        self.logger.info(f"Inicializando ConversationGenerator con modelo: {model}")
        
        # Inicializar formateador especializado para Mistral
        self.formatter = MistralPromptFormatter(self.logger)
        
        # Cargar configuraciones
        self.load_all_configs()

        # Añadir configuración de límites de respuesta
        self.response_limits = {
            'short': 100,  # caracteres
            'medium': 250,
            'long': 500
        }

        # Inicializar el sistema de gobierno
        self.governance = ConversationGovernance(
            self.types_config,
            self.stages_config
        )
        
    def _get_response_length_type(self, length):
        """Determina el tipo de longitud de respuesta"""
        if length <= self.response_limits['short']:
            return 'short'
        elif length <= self.response_limits['medium']:
            return 'medium'
        return 'long'
        
    def load_all_configs(self):
        """Carga y valida todas las configuraciones necesarias"""
        # Cargar configuraciones
        self.types_config = self.load_config('conversation_types.json')
        self.stages_config = self.load_config('conversation_stages.json')
        self.global_params = self.load_config('global_parameters.json')
        
        # Validar configuraciones
        self._validate_types_config(self.types_config)
        self._validate_stages_config(self.stages_config)
        self._validate_global_params(self.global_params)
        
        # Inicializar estado de la conversación
        self.conversation_history = []
        self.current_context = None
        self.current_type = None
        
        # Inicializar parámetros de estilo
        self.update_conversation_style()

    def _validate_types_config(self, config):
        """Valida la configuración específica de tipos de conversación"""
        if 'types' not in config:
            raise ValueError("Sección 'types' no encontrada en configuración")
            
        # Campos base requeridos para todos los tipos
        base_fields = ['system', 'context']
        
        for type_name, type_config in config['types'].items():
            # Validar campos base
            for field in base_fields:
                if field not in type_config:
                    raise ValueError(
                        f"Campo base requerido '{field}' no encontrado en types.{type_name}"
                    )
            
            # Validar que tenga al menos uno de los campos de enfoque
            focus_fields = ['focus', 'style', 'consideration', 'depth']
            if not any(field in type_config for field in focus_fields):
                raise ValueError(
                    f"El tipo {type_name} debe tener al menos uno de los campos: {focus_fields}"
                )
            
            # Validar que los campos sean listas
            for field in type_config:
                if field in ['system', 'user']:
                    continue
                if not isinstance(type_config[field], list):
                    raise ValueError(
                        f"El campo '{field}' en types.{type_name} debe ser una lista"
                    )
                if not type_config[field]:
                    raise ValueError(
                        f"El campo '{field}' en types.{type_name} no puede estar vacío"
                    )

    def _validate_stages_config(self, config):
        """Valida la configuración de etapas"""
        if 'stages' not in config:
            raise ValueError("Sección 'stages' no encontrada en configuración")
            
        required_fields = ['system', 'focus']
        for stage_name, stage_config in config['stages'].items():
            for field in required_fields:
                if field not in stage_config:
                    raise ValueError(
                        f"Campo requerido '{field}' no encontrado en stages.{stage_name}"
                    )
                if field != 'system' and not isinstance(stage_config[field], list):
                    raise ValueError(
                        f"El campo '{field}' en stages.{stage_name} debe ser una lista"
                    )

    def _validate_global_params(self, config):
        """Valida los parámetros globales"""
        required_fields = ['moods', 'personalities']
        for field in required_fields:
            if field not in config:
                raise ValueError(f"Campo requerido '{field}' no encontrado en parámetros globales")
            if not isinstance(config[field], list):
                raise ValueError(f"El campo '{field}' en parámetros globales debe ser una lista")

    def update_conversation_style(self):
        """Actualiza el estilo de la conversación seleccionando mood y personality aleatorios"""
        self.current_mood = random.choice(self.global_params["moods"])
        self.current_personality = random.choice(self.global_params["personalities"])
        self.logger.debug(f"Nuevo estilo: mood='{self.current_mood}', personality='{self.current_personality}'")
        
    def format_system_prompt(self, prompt_template):
        """Formatea un prompt del sistema reemplazando los placeholders de mood y personality"""
        return prompt_template.format(
            mood=self.current_mood,
            personality=self.current_personality
        )
        
        # Estado de la conversación
        self.conversation_history = []
        self.current_context = None
        
        self.logger.info("Inicialización completada")

    def setup_logging(self, debug):
        """Configura el sistema de logging con separación clara de concerns"""
        os.makedirs(self.logs_dir, exist_ok=True)
        
        # Logger principal
        self.logger = logging.getLogger('ConversationGenerator')
        self.logger.setLevel(logging.DEBUG if debug else logging.INFO)
        if self.logger.hasHandlers():
            self.logger.handlers.clear()

        # Logger para conversaciones
        self.conversation_logger = logging.getLogger('ConversationGenerator.conversation')
        self.conversation_logger.setLevel(logging.DEBUG)
        
        # Logger para clasificaciones
        self.classification_logger = logging.getLogger('ConversationGenerator.classification')
        self.classification_logger.setLevel(logging.DEBUG)

        # Handler para logs generales
        general_handler = RotatingFileHandler(
            os.path.join(self.logs_dir, 'generator.log'),
            maxBytes=10*1024*1024,
            backupCount=5
        )
        general_handler.setLevel(logging.INFO)
        general_formatter = logging.Formatter(
            '%(asctime)s - %(levelname)s - %(message)s'
        )
        general_handler.setFormatter(general_formatter)
        
        # Handler para logs de conversación
        conversation_handler = RotatingFileHandler(
            os.path.join(self.logs_dir, 'conversation.log'),
            maxBytes=10*1024*1024,
            backupCount=5
        )
        conversation_handler.setLevel(logging.DEBUG)
        conversation_formatter = logging.Formatter(
            '%(asctime)s\nCONVERSATION:\nSystem: %(system_prompt)s\nUser: %(user_prompt)s\nResponse: %(response)s\n' + '-'*80
        )
        conversation_handler.setFormatter(conversation_formatter)
        
        # Handler para logs de clasificación
        classification_handler = RotatingFileHandler(
            os.path.join(self.logs_dir, 'classification.log'),
            maxBytes=10*1024*1024,
            backupCount=5
        )
        classification_handler.setLevel(logging.DEBUG)
        classification_formatter = logging.Formatter(
            '%(asctime)s\nCLASSIFICATION:\nMessage: %(message)s\nResult: %(result)s\n' + '-'*80
        )
        classification_handler.setFormatter(classification_formatter)

        # Asignar handlers
        self.logger.addHandler(general_handler)
        self.conversation_logger.addHandler(conversation_handler)
        self.classification_logger.addHandler(classification_handler)

    def log_conversation(self, system_prompt, user_prompt, response):
        """Método específico para logging de conversaciones"""
        if not hasattr(self, 'conversation_logger'):
            return
            
        self.conversation_logger.info(
            '',
            extra={
                'system_prompt': self._clean_prompt(system_prompt),
                'user_prompt': self._clean_prompt(user_prompt),
                'response': self._clean_prompt(response)
            }
        )

    def log_classification(self, message_text, result):
        """Método específico para logging de clasificaciones"""
        if not hasattr(self, 'classification_logger'):
            return
            
        self.classification_logger.info(
            '',
            extra={
                'classification_input': self._clean_prompt(message_text),
                'classification_result': self._clean_prompt(result)
            }
        )
        
    def _clean_prompt(self, text):
        """Limpia y formatea el texto para logging"""
        if text is None:
            return ''
        return str(text).strip().replace('\n', ' ')
    
    def log_prompt(self, system_prompt, user_prompt):
        """Método específico para logging de prompts"""
        self.prompt_logger.debug(
            f"System Prompt:\n{system_prompt}\n\n"
            f"User Prompt:\n{user_prompt}"
        )
        
    def load_config(self, filename):
        config_path = os.path.join(self.config_dir, filename)
        self.logger.info(f"Cargando configuración desde: {config_path}")
        try:
            with open(config_path, 'r', encoding='utf-8') as file:
                config = json.load(file)
                self.logger.debug(f"Configuración cargada: {json.dumps(config, indent=2)}")
                return config
        except Exception as e:
            self.logger.error(f"Error al cargar configuración {filename}: {str(e)}")
            raise

    def get_current_personality_config(self):
        """Obtiene la configuración actual de personalidad"""
        return f"{self.current_mood}{self.current_personality}".strip()
        
    def extract_response(self, api_result):
        """Extrae y limpia la respuesta del resultado de la API"""
        if not api_result or 'choices' not in api_result:
            return None
            
        response = api_result['choices'][0].get('text', '')
        if not response:
            response = api_result['choices'][0].get('message', {}).get('content', '')
            
        return response.strip()
        
    def call_conversation_llm(self, system_prompt, user_prompt, temperature=0.7, max_tokens=150):
        """
        Método específico para llamadas LLM relacionadas con la generación de conversación.
        Esta función prioriza la coherencia y naturalidad de las respuestas.
        """
        try:
            # Formatear el prompt para conversación
            formatted_prompt = self.formatter.format_conversation(
                messages=[
                    {"role": "system", "content": system_prompt},
                    *self.conversation_history,
                    {"role": "user", "content": user_prompt}
                ],
                personality_config=self.get_current_personality_config()
            )
    
            self.logger.debug("Realizando llamada LLM para conversación")
            response = requests.post(
                self.api_url,
                headers={"Authorization": f"Bearer {self.api_key}"},
                json={
                    "model": self.model,
                    "prompt": formatted_prompt,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "stop": ["[INST]", "</s>"]
                }
            )
            
            if response.status_code == 200:
                result = response.json()
                parsed_response = self.extract_response(result)
                
                if parsed_response:
                    self.log_conversation(system_prompt, user_prompt, parsed_response)
                    return parsed_response
            else:
                self.logger.error(f"Error en API de conversación: {response.status_code}")
                return None
        except Exception as e:
            self.logger.error(f"Error en llamada LLM de conversación: {str(e)}")
            return None

    def call_classification_llm(self, message, previous_stage=None):
        """
        Método específico para llamadas LLM relacionadas con la clasificación.
        Esta función prioriza la precisión y consistencia en la clasificación.
        """
        context_prompt = (
            "INSTRUCCIÓN: Clasifica el siguiente mensaje según su tipo y etapa.\n\n"
            f"MENSAJE: {message}\n"
            f"ETAPA ANTERIOR: {previous_stage or 'ninguna'}\n\n"
            f"TIPOS VÁLIDOS: {', '.join(self.types_config['types'].keys())}\n"
            f"ETAPAS VÁLIDAS: {', '.join(self.stages_config['stages'].keys())}\n\n"
            "FORMATO REQUERIDO: JSON con 'type' y 'stage'\n"
            "EJEMPLO: {\"type\": \"informative_analytical\", \"stage\": \"development\"}"
        )

        try:
            self.logger.debug("Making classification LLM call")
            response = requests.post(
                self.api_url,
                headers={"Authorization": f"Bearer {self.api_key}"},
                json={
                    "model": self.model,
                    "prompt": context_prompt,
                    "temperature": 0.2,  # Temperatura baja para clasificación
                    "max_tokens": 30,    # Respuesta corta para JSON
                    "stop": ["[INST]", "</s>"]
                }
            )
            
            if response.status_code == 200:
                result = response.json()
                parsed_response = self.extract_response(result)
                
                if parsed_response:
                    # Limpiar y validar el JSON
                    result = parsed_response.strip()
                    if not result.startswith('{'):
                        result = result[result.find('{'):result.rfind('}')+1]
                        
                    classification = json.loads(result)
                    
                    # Validar la clasificación
                    if (classification["type"] in self.types_config["types"] and 
                        classification["stage"] in self.stages_config["stages"]):
                        
                        self.log_classification(message, json.dumps(classification))
                        return classification["type"], classification["stage"]
                        
            raise ValueError("Invalid classification response")
            
        except Exception as e:
            self.logger.error(f"Classification error: {str(e)}")
            return None

    def determine_message_type_and_stage(self, message, previous_stage=None):
        """Determina el tipo y etapa del mensaje"""
        # Si no hay tipo actual, usar uno por defecto
        if not self.current_type:
            self.current_type = list(self.types_config["types"].keys())[0]
        
        current_stage = previous_stage or 'initiation'
        
        try:
            # Obtener intent mediante LLM
            classification_result = self.call_classification_llm(message, previous_stage)
            
            if not classification_result:
                self.logger.warning("No se pudo obtener intent, usando valores por defecto")
                return self.current_type, self._determine_next_stage(current_stage)
                
            conv_type, stage = classification_result
            
            # Validar que el tipo existe
            if conv_type not in self.types_config["types"]:
                self.logger.warning(f"Tipo {conv_type} no válido, usando {self.current_type}")
                conv_type = self.current_type
                
            # Validar que la etapa existe
            if stage not in self.stages_config["stages"]:
                self.logger.warning(f"Etapa {stage} no válida, usando etapa siguiente")
                stage = self._determine_next_stage(current_stage)
            
            return conv_type, stage
            
        except Exception as e:
            self.logger.error(f"Error en determine_message_type_and_stage: {str(e)}")
            return self.current_type, self._determine_next_stage(current_stage)
                
    def _determine_next_stage(self, previous_stage):
        stages = ["initiation", "development", "consolidation", "transition"]
        if not previous_stage or previous_stage not in stages:
            return "initiation"
        current_index = stages.index(previous_stage)
        next_index = (current_index + 1) % len(stages)
        return stages[next_index]
        
    def generate_conversation_turn(self, user_message, conv_type, stage):
        """Genera un turno de conversación usando la configuración existente"""
        self.logger.info(f"Generando turno de conversación - Tipo: {conv_type}, Etapa: {stage}")
        
        try:
            # Validaciones
            if conv_type not in self.types_config["types"]:
                self.logger.warning(f"Tipo {conv_type} no encontrado, usando: {self.current_type}")
                conv_type = self.current_type
                
            if stage not in self.stages_config["stages"]:
                self.logger.warning(f"Etapa {stage} no encontrada, usando: initiation")
                stage = "initiation"
                
            # Obtener configuraciones
            type_config = self.types_config["types"][conv_type]
            stage_config = self.stages_config["stages"][stage]
            
            # Determinar qué campo usar para el enfoque del tipo
            focus_field = next((field for field in ['focus', 'style', 'consideration', 'depth'] 
                              if field in type_config), None)
                          
            if not focus_field:
                self.logger.warning(f"No se encontró campo de enfoque para tipo {conv_type}")
                focus = ""
            else:
                focus = random.choice(type_config[focus_field])
            
            stage_focus = random.choice(stage_config["focus"])
            
            # Actualizar estilo de conversación
            self.update_conversation_style()
            
            # Crear prompt del sistema con más contexto y restricciones
            system_prompt = f"""
            {type_config['system'].format(mood=self.current_mood, personality=self.current_personality)}
            {stage_config['system'].format(mood=self.current_mood, personality=self.current_personality)}
            
            Contexto actual: {self.current_context}
            Enfoque del tipo: {focus}
            Enfoque de la etapa: {stage_focus}
            
            IMPORTANTE: Mantén todas las respuestas estrictamente relacionadas con el tema de '{self.current_context}'.
            No te desvíes hacia otros temas o ejemplos no relacionados.
            """
        
            # Para generar el siguiente mensaje del usuario
            if user_message.startswith("Basado en la conversación anterior"):
                system_prompt = f"""
                Eres un usuario interesado en aprender sobre {self.current_context}.
                Genera una pregunta natural que siga el hilo de la conversación,
                manteniéndote estrictamente en el tema de {self.current_context}.
                No cambies a otros temas o ejemplos no relacionados.
                """
            
            # Ajustar tokens según etapa
            max_tokens = {
                'initiation': 100,
                'development': 150,
                'consolidation': 200,
                'transition': 100
            }.get(stage, 150)
            
            # Usar el nuevo método específico para conversación
            assistant_response = self.call_conversation_llm(
                system_prompt=system_prompt,
                user_prompt=user_message,
                max_tokens=max_tokens
            )
            
            if not assistant_response:
                self.logger.warning("Generando respuesta de fallback")
                assistant_response = (
                    "Lo siento, necesito más información para responder "
                    "adecuadamente en este contexto."
                )
            
            # Logging detallado
            if assistant_response:  # Evitar división por cero
                response_length = len(assistant_response)
                self.logger.info(
                    f"Respuesta generada:\n"
                    f"- Longitud: {response_length} caracteres\n"
                    f"- Tipo: {self._get_response_length_type(response_length)}\n"
                    f"- Contexto: {self.current_context}\n"
                    f"- Enfoque: {focus}"
                )
            
            return assistant_response
        except Exception as e:
            self.logger.error(f"Error en generate_conversation_turn: {str(e)}")
            return None

    def generate_initial_user_message(self):
        """Genera el mensaje inicial y establece el contexto de la conversación"""
        # Seleccionar tipo y contexto inicial
        self.current_type = random.choice(list(self.types_config["types"].keys()))
        self.current_context = random.choice(
            self.types_config["types"][self.current_type]["context"]
        )
        
        self.logger.info(
            f"Iniciando conversación - Tipo: {self.current_type}, "
            f"Contexto: {self.current_context}"
        )
        return f"Necesito ayuda con {self.current_context}"

    def generate_natural_conversation(self, num_turns=5):
        """Genera una conversación natural con el número especificado de turnos"""
        self.logger.info(f"Generando conversación natural de {num_turns} turnos")
        
        conversation_data = {
            "messages": [],
            "types": [],
            "stages": [],
            "message_lengths": []
        }
        
        try:
            user_message = self.generate_initial_user_message()
            if not user_message:
                raise ValueError("No se pudo generar mensaje inicial")
                
            prev_stage = None
            
            for turn in range(num_turns):
                self.logger.info(f"Generando turno {turn + 1}/{num_turns}")
                
                # Determinar tipo y etapa
                conv_type, stage = self.determine_message_type_and_stage(user_message, prev_stage)
                if not conv_type or not stage:
                    raise ValueError("Error en clasificación de mensaje")
                
                # Generar respuesta del asistente
                assistant_response = self.generate_conversation_turn(user_message, conv_type, stage)
                if not assistant_response:
                    raise ValueError("No se pudo generar respuesta del asistente")
                
                # Registrar longitudes de mensajes
                conversation_data["message_lengths"].extend([
                    len(user_message),
                    len(assistant_response)
                ])
                
                # Actualizar historia de la conversación
                self.conversation_history.extend([
                    {"role": "user", "content": user_message},
                    {"role": "assistant", "content": assistant_response}
                ])
                
                # Guardar datos de la conversación
                conversation_data["messages"].extend([user_message, assistant_response])
                conversation_data["types"].extend([conv_type, conv_type])
                conversation_data["stages"].extend([stage, stage])
                
                # Logging de estadísticas de longitud
                self.logger.info(
                    f"Estadísticas del turno {turn + 1}:\n"
                    f"- Longitud mensaje usuario: {len(user_message)}\n"
                    f"- Longitud respuesta asistente: {len(assistant_response)}"
                )
                
                # Generar siguiente mensaje del usuario usando call_conversation_llm
                user_prompt = """
                Basado en la conversación anterior, genera un siguiente mensaje natural del usuario
                que mantenga coherencia con el contexto y flujo de la conversación.
                Solo responde con el mensaje, sin explicaciones adicionales.
                """
                user_message = self.call_conversation_llm(
                    system_prompt="Eres un usuario manteniendo una conversación.",
                    user_prompt=user_prompt
                )
                
                if not user_message:
                    raise ValueError("No se pudo generar siguiente mensaje de usuario")
                    
                prev_stage = stage
                
        except Exception as e:
            self.logger.error(f"Error en generate_natural_conversation: {str(e)}")
            return conversation_data
            
        # Logging de estadísticas finales
        msg_lengths = conversation_data["message_lengths"]
        if msg_lengths:  # Evitar división por cero
            self.logger.info(
                f"Estadísticas finales de la conversación:\n"
                f"- Promedio longitud mensajes: {sum(msg_lengths)/len(msg_lengths):.2f}\n"
                f"- Mensaje más corto: {min(msg_lengths)}\n"
                f"- Mensaje más largo: {max(msg_lengths)}"
            )
        
        self.logger.info(f"Conversación generada con {len(conversation_data['messages'])} mensajes")
        return conversation_data

    def generate_dataset(self, num_conversations=1):
        self.logger.info(f"Iniciando generación de dataset con {num_conversations} conversaciones")
        
        all_data = {
            "message": [],
            "conversation_id": [],
            "message_id": [],
            "role": [],
            "type": [],
            "stage": []
        }
        
        for conv_id in tqdm(range(num_conversations)):
            self.logger.info(f"Generando conversación {conv_id + 1}/{num_conversations}")
            self.conversation_history = []
            
            try:
                conv_data = self.generate_natural_conversation()
                
                for msg_id, (message, type_, stage_) in enumerate(zip(
                    conv_data["messages"],
                    conv_data["types"],
                    conv_data["stages"]
                )):
                    all_data["message"].append(message)
                    all_data["conversation_id"].append(conv_id)
                    all_data["message_id"].append(msg_id)
                    all_data["role"].append("user" if msg_id % 2 == 0 else "assistant")
                    all_data["type"].append(type_)
                    all_data["stage"].append(stage_)
                    
            except Exception as e:
                self.logger.error(f"Error en conversación {conv_id}: {str(e)}")
                continue
        
        df = pd.DataFrame(all_data)
        self.logger.info(f"Dataset generado con {len(df)} mensajes")
        return df

    def export_dataset(self, df, format='both'):
        """
        Exporta el dataset en los formatos especificados
        Args:
            df: DataFrame con los datos
            format: 'csv', 'json', o 'both'
        """
        self.logger.info(f"Exportando dataset en formato: {format}")
        timestamp = datetime.now().strftime("%y-%m-%d_%H-%M-%S")
        
        try:
            if format in ['csv', 'both']:
                csv_path = os.path.join(self.generated_dir, f'conversations_{timestamp}.csv')
                df.to_csv(csv_path, index=False)
                self.logger.info(f"Dataset exportado a CSV: {csv_path}")
                
            if format in ['json', 'both']:
                # Convertir DataFrame a diccionario y asegurar tipos serializables
                df_dict = df.copy()
                # Convertir columnas numéricas a int/float Python estándar
                df_dict['conversation_id'] = df_dict['conversation_id'].astype(int)
                df_dict['message_id'] = df_dict['message_id'].astype(int)
                
                conversations = []
                for conv_id in df_dict['conversation_id'].unique():
                    conv_df = df_dict[df_dict['conversation_id'] == conv_id]
                    conversation = {
                        "id": int(conv_id),
                        "messages": []
                    }
                    
                    for _, row in conv_df.iterrows():
                        message = {
                            "id": int(row['message_id']),
                            "role": row['role'],
                            "content": row['message'],
                            "metadata": {
                                "type": row['type'],
                                "stage": row['stage']
                            }
                        }
                        conversation["messages"].append(message)
                    
                    conversations.append(conversation)
                
                json_path = os.path.join(self.generated_dir, f'conversations_{timestamp}.json')
                with open(json_path, 'w', encoding='utf-8') as f:
                    json.dump(conversations, f, indent=2, ensure_ascii=False)
                self.logger.info(f"Dataset exportado a JSON: {json_path}")
                
        except Exception as e:
            self.logger.error(f"Error al exportar dataset: {str(e)}")
            raise

def main():
    # Configuración
    MODEL = "mistral-7b-instruct"
    API_URL = "https://mistral-7b-instruct-v0-3-maas-apicast-production.apps.prod.rhoai.rh-aiservices-bu.com:443/v1/completions"
    API_KEY = "a9513a7c603fca0ea92c1ee60c410cac"
    
    # Inicializar generador con debugging activado
    generator = ConversationGenerator(MODEL, API_URL, API_KEY, debug=True)
    
    # Generar dataset
    df = generator.generate_dataset(num_conversations=1)
    
    # Exportar datos
    generator.export_dataset(df)

if __name__ == "__main__":
    main()

  0%|                                                                                           | 0/1 [00:00<?, ?it/s]--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib64/python3.11/logging/__init__.py", line 449, in format
    return self._format(record)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.11/logging/__init__.py", line 445, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
KeyError: 'result'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib64/python3.11/logging/handlers.py", line 73, in emit
    if self.shouldRollover(record):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.11/logging/handlers.py", line 196, in shouldRollover
    msg = "%s\n" % self.format(record)
                   ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.11/logging/__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  